In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, group
from pyspark.sql import functions as F

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat, src = cfg["catalog"], group.source(cfg, env)
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
f = lambda t: spark.table(f"{src}.gold.{t}")
print(cat, "group customers, FQF from", src)

In [0]:
mp, dim, rev = g("map_group_customer"), g("dim_group_customer"), g("group_match_review")
pat = g("dim_patient").filter("is_current AND patient_key <> -1").select(F.lit("bhg").alias("company_code"),
        F.col("patient_id").alias("member_id"), "id_fingerprint")
cus = f("dim_customer").filter("is_current AND customer_key <> -1 AND customer_id = master_customer_id").select(
        F.lit("fqf").alias("company_code"), F.col("master_customer_id").alias("member_id"), "id_fingerprint")
src_members = pat.unionByName(cus)

n_src, n_map = src_members.count(), mp.count()
check("every current patient and master customer is mapped once",
      n_src == n_map and mp.select("company_code", "member_id").distinct().count() == n_map, f"{n_map} of {n_src}")
check("person key filled and unique in dim_group_customer",
      dim.filter("group_customer_key IS NULL").count() == 0 and dim.select("group_customer_key").distinct().count() == dim.count())
check("exactly one unknown person (-1)", dim.filter("group_customer_key = -1").count() == 1)
check("every mapped member has its person", mp.join(dim, "group_customer_key", "left_anti").count() == 0)

two = (mp.join(src_members, ["company_code", "member_id"]).filter("id_fingerprint IS NOT NULL")
         .groupBy("group_customer_key").agg(F.countDistinct("id_fingerprint").alias("n")).filter("n > 1").count())
check("no person holds two different Aadhaar fingerprints", two == 0, two)
same_fp = (src_members.filter("id_fingerprint IS NOT NULL").join(mp, ["company_code", "member_id"])
             .groupBy("id_fingerprint").agg(F.countDistinct("group_customer_key").alias("n")).filter("n > 1").count())
check("one Aadhaar fingerprint = one person", same_fp == 0, same_fp)

shared = dim.filter("is_shared").count()
check("shared flag = in both companies", shared == dim.filter("is_bhg_patient AND is_fqf_customer").count())
print("INFO persons:", dim.count() - 1, "| shared by both companies:", shared)
dim.groupBy("match_method").count().orderBy("match_method").show()

links = group.invoice_links(g, f)
lk = (links.join(mp.filter("company_code = 'bhg'").select(F.col("member_id").alias("patient_id"), F.col("group_customer_key").alias("pk")), "patient_id")
           .join(mp.filter("company_code = 'fqf'").select(F.col("member_id").alias("customer_id"), F.col("group_customer_key").alias("ck")), "customer_id"))
loose = lk.filter("pk <> ck").join(rev.select(F.col("bhg_patient_id").alias("patient_id"), F.col("fqf_customer_id").alias("customer_id")),
                                   ["patient_id", "customer_id"], "left_anti").count()
check("every hospital-bill link is one person or sent to review", loose == 0, loose)
print("INFO hospital-bill links:", links.count())

check("review never lists a pair that is already one person",
      rev.join(mp.filter("company_code = 'bhg'").select(F.col("member_id").alias("bhg_patient_id"), F.col("group_customer_key").alias("pk")), "bhg_patient_id")
         .join(mp.filter("company_code = 'fqf'").select(F.col("member_id").alias("fqf_customer_id"), F.col("group_customer_key").alias("ck")), "fqf_customer_id")
         .filter("pk = ck").count() == 0)
check("review status is open or decided", rev.filter("status NOT IN ('open', 'decided')").count() == 0)
print("INFO review rows:", rev.count())

bad = [c for t in ("map_group_customer", "dim_group_customer", "group_match_review")
       for c in g(t).columns if c in ("full_name", "phone", "aadhaar", "dob", "contact_fingerprint")]
check("no personal columns in group tables", not bad, bad)
m = [r[0] for r in spark.sql(f"""SELECT mask_name FROM {cat}.information_schema.column_masks
        WHERE table_schema = 'gold' AND table_name = 'dim_group_customer' AND column_name = 'id_fingerprint'""").collect()]
check("dim_group_customer.id_fingerprint masked with mask_token", m == [f"{cat}.ops.mask_token"], m)
check("decisions table exists", spark.catalog.tableExists(f"{cat}.ops.group_match_decisions"))

print(f"\nt22_group_customer: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t22_group_customer failed")